In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

PROJECT_PATH = '/content/drive/MyDrive/DataMining_Project'

data_path = f'{PROJECT_PATH}/data/Womens Clothing E-Commerce Reviews.csv'
df = pd.read_csv(data_path)

print("Orijinal Veri Seti Boyutu:", df.shape)

if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

print("\nTemizlik Öncesi Eksik Veriler:\n", df.isnull().sum())

df = df.dropna(subset=['Review Text'])

df['Title'] = df['Title'].fillna("")

df['Division Name'] = df['Division Name'].fillna("Unknown")
df['Department Name'] = df['Department Name'].fillna("Unknown")
df['Class Name'] = df['Class Name'].fillna("Unknown")

print("\nTemizlik Sonrası Eksik Veriler:\n", df.isnull().sum())
print("\nTemizlenmiş Veri Seti Boyutu:", df.shape)

clean_data_path = f'{PROJECT_PATH}/data/Cleaned_ECommerce_Reviews.csv'
df.to_csv(clean_data_path, index=False)

print(f"\nVeri kaydedildi: {clean_data_path}")

Orijinal Veri Seti Boyutu: (23486, 11)

Temizlik Öncesi Eksik Veriler:
 Clothing ID                   0
Age                           0
Title                      3810
Review Text                 845
Rating                        0
Recommended IND               0
Positive Feedback Count       0
Division Name                14
Department Name              14
Class Name                   14
dtype: int64

Temizlik Sonrası Eksik Veriler:
 Clothing ID                0
Age                        0
Title                      0
Review Text                0
Rating                     0
Recommended IND            0
Positive Feedback Count    0
Division Name              0
Department Name            0
Class Name                 0
dtype: int64

Temizlenmiş Veri Seti Boyutu: (22641, 10)

Veri kaydedildi: /content/drive/MyDrive/DataMining_Project/data/Cleaned_ECommerce_Reviews.csv


In [3]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

PROJECT_PATH = '/content/drive/MyDrive/DataMining_Project'
df = pd.read_csv(f'{PROJECT_PATH}/data/Cleaned_ECommerce_Reviews.csv')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def process_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(cleaned_words)

print("NLP işlemi başlıyor...")
df['Processed_Review'] = df['Review Text'].apply(process_text)

print(df[['Review Text', 'Processed_Review']].head())

output_path = f'{PROJECT_PATH}/data/NLP_Processed_Reviews.csv'
df.to_csv(output_path, index=False)
print(f"NLP işlemi tamamlandı. Şuraya kaydedildi: {output_path}")

NLP işlemi başlıyor...
                                         Review Text  \
0  Absolutely wonderful - silky and sexy and comf...   
1  Love this dress!  it's sooo pretty.  i happene...   
2  I had such high hopes for this dress and reall...   
3  I love, love, love this jumpsuit. it's fun, fl...   
4  This shirt is very flattering to all due to th...   

                                    Processed_Review  
0        absolutely wonderful silky sexy comfortable  
1  love dress sooo pretty happened find store im ...  
2  high hope dress really wanted work initially o...  
3  love love love jumpsuit fun flirty fabulous ev...  
4  shirt flattering due adjustable front tie perf...  
NLP işlemi tamamlandı. Şuraya kaydedildi: /content/drive/MyDrive/DataMining_Project/data/NLP_Processed_Reviews.csv


In [4]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon', quiet=True)

PROJECT_PATH = '/content/drive/MyDrive/DataMining_Project'
df = pd.read_csv(f'{PROJECT_PATH}/data/NLP_Processed_Reviews.csv')

sia = SentimentIntensityAnalyzer()

def get_sentiment_score(text):
    if pd.isna(text) or str(text).strip() == "":
        return 0.0
    return sia.polarity_scores(str(text))['compound']

print("Duygu analizi yapılıyor ve yeni özellikler türetiliyor...")

df['Sentiment_Score'] = df['Processed_Review'].apply(get_sentiment_score)

df['Word_Count'] = df['Processed_Review'].apply(lambda x: len(str(x).split()))

print("\nTüretilen Yeni Özellikler:")
print(df[['Processed_Review', 'Sentiment_Score', 'Word_Count']].head())

output_path = f'{PROJECT_PATH}/data/Feature_Engineered_Reviews.csv'
df.to_csv(output_path, index=False)
print(f"\nYeni veri seti kaydedildi: {output_path}")

Duygu analizi yapılıyor ve yeni özellikler türetiliyor...

Türetilen Yeni Özellikler:
                                    Processed_Review  Sentiment_Score  \
0        absolutely wonderful silky sexy comfortable           0.8991   
1  love dress sooo pretty happened find store im ...           0.9710   
2  high hope dress really wanted work initially o...           0.9081   
3  love love love jumpsuit fun flirty fabulous ev...           0.9437   
4  shirt flattering due adjustable front tie perf...           0.9062   

   Word_Count  
0           5  
1          30  
2          48  
3          14  
4          16  

Yeni veri seti kaydedildi: /content/drive/MyDrive/DataMining_Project/data/Feature_Engineered_Reviews.csv


In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

PROJECT_PATH = '/content/drive/MyDrive/DataMining_Project'
df = pd.read_csv(f'{PROJECT_PATH}/data/Feature_Engineered_Reviews.csv')

tfidf = TfidfVectorizer(max_features=1000)

print("Kelimeler matematiksel vektörlere dönüştürülüyor...")

tfidf_matrix = tfidf.fit_transform(df['Processed_Review'].fillna(""))

tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())

final_df = pd.concat([df, tfidf_df], axis=1)

print("\nGüncel Veri Seti Boyutu:", final_df.shape)

output_path = f'{PROJECT_PATH}/data/Final_Machine_Learning_Data.csv'
final_df.to_csv(output_path, index=False)

joblib.dump(tfidf, f'{PROJECT_PATH}/src/tfidf_vectorizer.pkl')

print(f"\nTüm veriler başarıyla birleştirildi ve şuraya kaydedildi: {output_path}")

Kelimeler matematiksel vektörlere dönüştürülüyor...

Güncel Veri Seti Boyutu: (22641, 1013)

Tüm veriler başarıyla birleştirildi ve şuraya kaydedildi: /content/drive/MyDrive/DataMining_Project/data/Final_Machine_Learning_Data.csv
